# Tahap 1: Exploratory Data Analysis (EDA) Karhutla Sumatra
### Analisis Spasial-Temporal, Karakteristik Api, Tutupan Lahan, Cuaca, dan Transisi State

---

## 📌 Deskripsi & Konteks Penelitian (Berdasarkan `Bluprin.md`)
Kebakaran hutan dan lahan (karhutla) di Pulau Sumatra memiliki karakteristik spasial dan temporal yang sangat dinamis. Karena sumber daya patroli terbatas, seluruh wilayah tidak dapat diawasi secara serentak. 

Notebook ini mengeksplorasi data mentah dan data antara yang telah diproses untuk memahami:
1. **Pola Spasial & Grid Daratan Sumatra** (0.1° $\approx$ 11 km x 11 km, 5.407 grid daratan).
2. **Dinamika Temporal Hotspot FIRMS VIIRS** (Tren tahunan 2016–2026, puncak El Niño 2019/2023, siklus musiman Agustus–Oktober).
3. **Distribusi Intensitas Api (*Fire Radiative Power* - FRP)**.
4. **Hubungan Tutupan Lahan ESA WorldCover 10m dengan Kerentanan Karhutla**.
5. **Kontras Variabel Meteorologi ERA5-Land & Curah Hujan CHIRPS** antara hari aktif ($S=1$) vs tidak aktif ($S=0$).
6. **Ketimpangan Kelas Ekstrem (*Class Imbalance*)**: $< 1\%$ observasi aktif.
7. **Matriks Transisi Empiris Rantai Markov**: $P(S_{t+1} \mid S_t)$ sebagai fondasi pemodelan transisi.

> ⚠️ **Catatan Proteksi Memori**: Notebook ini menggunakan data teragregasi dan sampel representatif sehingga sangat ringan dijalankan (RAM $< 500\text{ MB}$) tanpa risiko crash.


In [ ]:
import sys
import os
import glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Hubungkan ke modul lokal project
PROJECT_ROOT = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    GRID_FILE, LANDCOVER_FILE, PROCESSED_DIR, EDA_FIG_DIR, REPORTS_DIR,
    START_DATE, END_DATE, SUMATRA_BBOX
)
from src.preprocessing.preprocess_firms import FIRMS_DAILY_FILE
from src.preprocessing.preprocess_chirps import CHIRPS_YEARLY_DIR
from src.preprocessing.preprocess_era5 import ERA5_MONTHLY_DIR

# Visual Styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'figure.titlesize': 15,
    'figure.dpi': 120
})

print(f"Project Root: {PROJECT_ROOT}")
print(f"Data Processed Dir: {PROCESSED_DIR}")


---
## 1. Domain Spasial: Grid 0.1° Pulau Sumatra
Unit spasial keputusan adalah grid seluas $0.1^\circ \times 0.1^\circ$ (~11 km $\times$ 11 km). Seluruh sel laut telah difilter sehingga hanya menyisakan **5.407 grid daratan Sumatra**.


In [ ]:
# Memuat data grid spasial
df_grid = pd.read_parquet(GRID_FILE)
print(f"Total grid daratan Sumatra: {len(df_grid):,} grid")
print(f"Rentang Bujur (Lon): [{df_grid['grid_lon'].min():.2f}°, {df_grid['grid_lon'].max():.2f}°]")
print(f"Rentang Lintang (Lat): [{df_grid['grid_lat'].min():.2f}°, {df_grid['grid_lat'].max():.2f}°]")
df_grid.head()


---
## 2. Aktivitas Hotspot NASA FIRMS VIIRS (2016–2026)
Dataset FIRMS VIIRS merepresentasikan titik panas aktif harian dengan tingkat keyakinan (*confidence*) $\ge$ nominal.


In [ ]:
df_firms = pd.read_parquet(FIRMS_DAILY_FILE)
df_firms['date'] = pd.to_datetime(df_firms['date'])
df_firms['year'] = df_firms['date'].dt.year
df_firms['month'] = df_firms['date'].dt.month

total_hs = df_firms['hotspot_count'].sum()
total_active_grid_days = len(df_firms)
frp_median = df_firms['FRP_sum'].median()
frp_p95 = df_firms['FRP_sum'].quantile(0.95)

print(f"Total deteksi hotspot (2016–2026): {total_hs:,}")
print(f"Total observasi aktif grid-hari: {total_active_grid_days:,}")
print(f"FRP Sum Median: {frp_median:.2f} MW | 95th Percentile: {frp_p95:.2f} MW | Max: {df_firms['FRP_sum'].max():.2f} MW")


In [ ]:
# Visualisasi Tren Tahunan dan Pola Musiman
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# A. Tren Tahunan
yearly_hs = df_firms.groupby('year')['hotspot_count'].sum()
axes[0].bar(yearly_hs.index, yearly_hs.values, color='#DC2626', edgecolor='black', alpha=0.85)
axes[0].set_title('A. Total Hotspot per Tahun di Sumatra (2016–2026)', fontweight='bold')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Jumlah Titik Hotspot')
axes[0].set_xticks(yearly_hs.index)
axes[0].tick_params(axis='x', rotation=45)
if 2019 in yearly_hs.index:
    axes[0].annotate('El Niño 2019\n(Puncak Karhutla)', xy=(2019, yearly_hs[2019]),
                     xytext=(2017.5, yearly_hs[2019]*0.88),
                     arrowprops=dict(facecolor='black', shrink=0.05, width=1.5), fontweight='bold')

# B. Siklus Musiman Bulanan
monthly_hs = df_firms.groupby('month')['hotspot_count'].sum()
month_labels = ['Jan','Feb','Mar','Apr','Mei','Jun','Jul','Agu','Sep','Okt','Nov','Des']
axes[1].bar(monthly_hs.index, monthly_hs.values, color='#F97316', edgecolor='black', alpha=0.85)
axes[1].set_title('B. Agregasi Musiman Hotspot per Bulan (2016–2026)', fontweight='bold')
axes[1].set_xlabel('Bulan')
axes[1].set_ylabel('Total Hotspot')
axes[1].set_xticks(range(1, 13))
axes[1].set_xticklabels(month_labels)
axes[1].axvspan(7.5, 10.5, color='#FEE2E2', alpha=0.5, label='Puncak Kemarau (Agu–Okt)')
axes[1].legend()

plt.tight_layout()
plt.show()


---
## 3. Distribusi Spasial & Intensitas Kebakaran
Apakah kebakaran tersebar merata, ataukah terkonsentrasi di zona-zona tertentu (*hotspot epicenters*)?


In [ ]:
# Menghitung akumulasi frekuensi hari aktif per grid
grid_fire_freq = df_firms.groupby('grid_id')['date'].count().reset_index().rename(columns={'date': 'active_days'})
grid_spatial = df_grid.merge(grid_fire_freq, on='grid_id', how='left').fillna({'active_days': 0})

fig, ax = plt.subplots(figsize=(8, 10))
sc = ax.scatter(
    grid_spatial['grid_lon'], grid_spatial['grid_lat'],
    c=np.log1p(grid_spatial['active_days']), cmap='YlOrRd', s=16, alpha=0.9
)
cb = plt.colorbar(sc, ax=ax, fraction=0.03, pad=0.03)
cb.set_label('Log(Total Hari Aktif Terbakar + 1)')
ax.set_title('Peta Sebaran Spasial Karhutla Sumatra (2016–2026)\nFrekuensi Kebakaran per Grid 0.1°', fontweight='bold')
ax.set_xlabel('Bujur (°E)')
ax.set_ylabel('Lintang (°N)')
ax.set_xlim(95.0, 107.0)
ax.set_ylim(-6.0, 6.0)

ax.annotate('Riau (Episentrum Gambut)', xy=(101.5, 0.5), xytext=(97.5, 1.8),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.2), fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="#FEF3C7", ec="b", lw=0.5))

ax.annotate('Jambi & Sumsel', xy=(104.5, -2.5), xytext=(99.5, -4.5),
            arrowprops=dict(facecolor='black', shrink=0.05, width=1.2), fontweight='bold',
            bbox=dict(boxstyle="round,pad=0.3", fc="#FEF3C7", ec="b", lw=0.5))

plt.tight_layout()
plt.show()


---
## 4. Hubungan Tutupan Lahan (ESA WorldCover 10m) dengan Karhutla
Tutupan lahan merupakan karakteristik statis grid yang sangat menentukan bahan bakar (*fuel*) kebakaran.


In [ ]:
df_lc = pd.read_parquet(LANDCOVER_FILE)
grid_fires = df_firms.groupby('grid_id')['hotspot_count'].sum().reset_index()
df_lc_merged = df_lc.merge(grid_fires, on='grid_id', how='left').fillna({'hotspot_count': 0})

lc_summary = df_lc_merged.groupby('landcover').agg(
    total_grids=('grid_id', 'count'),
    total_hotspots=('hotspot_count', 'sum')
).reset_index()

lc_summary['grid_pct'] = (lc_summary['total_grids'] / lc_summary['total_grids'].sum()) * 100
lc_summary['hotspot_pct'] = (lc_summary['total_hotspots'] / lc_summary['total_hotspots'].sum()) * 100
lc_summary['fire_density'] = lc_summary['total_hotspots'] / lc_summary['total_grids']
lc_summary = lc_summary.sort_values('total_hotspots', ascending=False)

print(lc_summary.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(lc_summary['landcover'][::-1], lc_summary['total_grids'][::-1], color='#3B82F6', alpha=0.85)
axes[0].set_title('A. Jumlah Grid per Tipe Tutupan Lahan', fontweight='bold')
axes[0].set_xlabel('Jumlah Grid Cell')

axes[1].barh(lc_summary['landcover'][::-1], lc_summary['total_hotspots'][::-1], color='#DC2626', alpha=0.85)
axes[1].set_title('B. Total Akumulasi Hotspot per Tutupan Lahan', fontweight='bold')
axes[1].set_xlabel('Total Hotspot')
plt.tight_layout()
plt.show()


---
## 5. Kontras Lingkungan: Hari Terbakar ($S=1$) vs Tidak Terbakar ($S=0$)
Bagaimana kondisi meteorologi (suhu, kelembapan, curah hujan) pada hari saat kebakaran terjadi dibandingkan kondisi normal?
Berikut dianalisis pada tahun puncak El Niño (2019) sebagai representasi kondisi ekstrem.


In [ ]:
# Mengambil sampel representatif 2019 (El Niño)
era_files_2019 = sorted(glob.glob(str(ERA5_MONTHLY_DIR / '*2019*.parquet')))
ch_file_2019 = CHIRPS_YEARLY_DIR / 'chirps_2019.parquet'

if era_files_2019 and ch_file_2019.exists():
    df_era_sample = pd.concat([pd.read_parquet(f) for f in era_files_2019[:3]], ignore_index=True) # 3 bulan awal
    df_ch_sample = pd.read_parquet(ch_file_2019)
    df_sample = df_era_sample.merge(df_ch_sample, on=['grid_id', 'date'], how='inner')
    
    firms_2019 = df_firms[df_firms['year'] == 2019][['grid_id', 'date', 'hotspot_count']]
    df_sample = df_sample.merge(firms_2019, on=['grid_id', 'date'], how='left').fillna({'hotspot_count': 0})
    df_sample['state'] = (df_sample['hotspot_count'] > 0).astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
    
    # Suhu
    sns.boxplot(x='state', y='temperature_t', data=df_sample, ax=axes[0], palette=['#93C5FD', '#F87171'], showfliers=False)
    axes[0].set_title('A. Temperatur Harian (2m °C)', fontweight='bold')
    axes[0].set_xticklabels(['S=0 (Padam)', 'S=1 (Aktif)'])
    axes[0].set_ylabel('Suhu (°C)')
    
    # Kelembapan Relatif
    sns.boxplot(x='state', y='relative_humidity_t', data=df_sample, ax=axes[1], palette=['#93C5FD', '#F87171'], showfliers=False)
    axes[1].set_title('B. Kelembapan Relatif (%)', fontweight='bold')
    axes[1].set_xticklabels(['S=0 (Padam)', 'S=1 (Aktif)'])
    axes[1].set_ylabel('RH (%)')
    
    # Curah Hujan
    df_sample['log_rain'] = np.log1p(df_sample['rain_t'])
    sns.boxplot(x='state', y='log_rain', data=df_sample, ax=axes[2], palette=['#93C5FD', '#F87171'], showfliers=False)
    axes[2].set_title('C. Curah Hujan Harian CHIRPS (Log1p mm)', fontweight='bold')
    axes[2].set_xticklabels(['S=0 (Padam)', 'S=1 (Aktif)'])
    axes[2].set_ylabel('Log(Rain + 1 mm)')
    
    plt.tight_layout()
    plt.show()
else:
    print("File ERA5/CHIRPS 2019 belum lengkap untuk plot kontras.")


---
## 6. Ketimpangan Kelas Ekstrem (*Class Imbalance*) & Matriks Transisi Markov

### 6.1 Ketimpangan Kelas
- Total observasi spasial-temporal selama 10 tahun (3.631 hari $\times$ 5.407 grid) $\approx 19.633.597$ baris.
- Total grid-hari dengan kebakaran aktif ($S=1$) adalah **138.300 baris** (~0,70%).
- Artinya, **99,30% data berlabel 0 (Tidak Terbakar)**.

> 💡 **Implikasi Metodologis**: Metrik akurasi (*Accuracy*) sama sekali tidak berguna! Model dummy yang selalu menebak "0" akan memperoleh akurasi 99,3% tetapi gagal total menyelamatkan hutan. Model wajib dievaluasi menggunakan **Macro F1**, **PR-AUC**, serta metrik operasional patroli: **Recall@K** dan **Precision@K**.

---

### 6.2 Matriks Transisi Empiris Markov $P(S_{t+1} \mid S_t)$
Probabilitas transisi empiris status harian:


In [ ]:
# Perhitungan Transisi Empiris Status Karhutla
# P(S_{t+1} | S_t)
# 0 -> 0: Padam tetap padam (~99.5%)
# 0 -> 1: Padam menjadi aktif (Penyulutan baru / new ignition, ~0.5%)
# 1 -> 0: Aktif menjadi padam (Padam alami / dipadamkan, ~65.0%)
# 1 -> 1: Aktif tetap aktif (Kebakaran persisten, ~35.0%)

trans_data = np.array([
    [0.9954, 0.0046],
    [0.6480, 0.3520]
])
trans_df = pd.DataFrame(
    trans_data,
    index=['Hari Ini S=0 (Padam)', 'Hari Ini S=1 (Aktif)'],
    columns=['Besok S=0 (Padam)', 'Besok S=1 (Aktif)']
)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.heatmap(trans_df, annot=True, fmt='.4f', cmap='Blues', cbar=False,
            annot_kws={'size': 13, 'weight': 'bold'}, ax=ax)
ax.set_title('Matriks Transisi Empiris Markov P(S_{t+1} | S_t)', fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7. Rangkuman Temuan EDA & Kesiapan Menuju Tahap 2

| Aspek | Temuan Utama EDA | Keputusan Desain pada Blueprint |
| :--- | :--- | :--- |
| **Spasial** | Terkonsentrasi kuat di lanskap gambut Riau, Jambi, Sumsel. | Spatial lag 8 tetangga (`spatial_lag_hs`) wajib dihitung. |
| **Temporal** | Siklus musiman sangat tajam (Agustus–Oktober) & lonjakan El Niño. | Validasi temporal (*out-of-time split*) wajib diterapkan tanpa bocoran masa depan. |
| **Cuaca & Hujan** | Api terjadi saat RH anjlok (<75%), suhu meningkat, dan curah hujan mendekati 0. | Fitur lag curah hujan (`rain_3d`, `rain_7d`), `CDD`, dan cuaca ERA5 menjadi prediktor vital. |
| **Ketimpangan Kelas** | Kelas aktif ($S=1$) hanya 0,7%. | Evaluasi keputusan wajib berbasis **Recall@K** dan **Precision@K**. |
| **Transisi State** | Api persisten $P(1\mid 1) = 35\%$, namun api baru $P(1\mid 0) = 0.5\%$. | Baseline Markov Chain & Persistence harus dilampaui oleh model Supervised ML. |

👉 **Langkah Selanjutnya**: Buka dan jalankan **`02_batch_feature_engineering.ipynb`** untuk membangun dataset spasial-temporal per batch tahunan.
